# Regression Evaluation Metrics: Mathematical Foundations and Error Analysis

**Session:** 2026-10-31 · Competition tasks and metrics  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

Regression tasks require predicting continuous target variables $y \in \mathbb{R}$. In competitive benchmarks and practical deployments, selecting the appropriate evaluation metric determines how models are ranked and how optimization trade-offs are navigated.

## Core Metrics Overview

| Metric | Formula | Units | Outlier Sensitivity |
| :--- | :--- | :--- | :--- |
| **Mean Squared Error (MSE)** | $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ | $\text{Units}^2$ | Very High (Quadratic) |
| **Root Mean Squared Error (RMSE)** | $\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$ | $\text{Units}$ | High (Quadratic penalty inside square root) |
| **Mean Absolute Error (MAE)** | $\frac{1}{n}\sum \|y_i - \hat{y}_i\|$ | $\text{Units}$ | Moderate (Linear) |
| **Mean Absolute Percentage Error (MAPE)** | $\frac{100\%}{n}\sum \|\frac{y_i - \hat{y}_i}{y_i}\|$ | Percentage | Asymmetric / Undefined at $y_i=0$ |
| **Coefficient of Determination ($R^2$)** | $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ | Dimensionless $[-\infty, 1]$ | Tied to Residual Sum of Squares |

## Learning Outcomes

- Implement all foundational regression metrics from mathematical first principles.
- Contrast the outlier sensitivity of MSE versus the robust median-seeking nature of MAE.
- Implement the Huber loss function as a smooth hybrid bridging MAE and MSE.
- Analyze residual distributions ($e_i = y_i - \hat{y}_i$) on `data/regression_house_prices.csv`.
- Evaluate and rank competing model predictions on real tabular validation data.


In [ ]:
import numpy as np


def mean_squared_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((y_true - y_pred) ** 2))


def root_mean_squared_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mean_absolute_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def mean_absolute_percentage_error(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    eps: float = 1e-8,
) -> float:
    """Compute MAPE with epsilon protection against division by zero."""
    denominator = np.where(np.abs(y_true) < eps, eps, np.abs(y_true))
    return float(np.mean(np.abs((y_true - y_pred) / denominator)) * 100.0)


def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute R^2 (Coefficient of determination)."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0.0:
        return 0.0
    return float(1.0 - (ss_res / ss_tot))


# Baseline sanity check
act = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
pred = np.array([1.1, 2.2, 2.9, 4.1, 4.8])

print(f"MSE:   {mean_squared_error(act, pred):.4f}")
print(f"RMSE:  {root_mean_squared_error(act, pred):.4f}")
print(f"MAE:   {mean_absolute_error(act, pred):.4f}")
print(f"MAPE:  {mean_absolute_percentage_error(act, pred):.2f}%")
print(f"R^2:   {r2_score(act, pred):.4f}")

## 1. Outlier Sensitivity Comparison: MSE vs MAE

Because MSE squares residuals ($e^2$), a single gross outlier distorts the metric disproportionately. MAE weights all error magnitudes linearly, reflecting the conditional median rather than the conditional mean.


In [ ]:
import numpy as np

# Clean dataset
y_clean = np.array([10.0, 11.0, 9.5, 10.5, 10.0])
y_est = np.array([10.1, 10.9, 9.7, 10.3, 10.1])

mse_clean = mean_squared_error(y_clean, y_est)
mae_clean = mean_absolute_error(y_clean, y_est)

# Introduce a single outlier into observation 5
y_corrupted = y_est.copy()
y_corrupted[-1] = 50.0  # Massive prediction explosion

mse_dirty = mean_squared_error(y_clean, y_corrupted)
mae_dirty = mean_absolute_error(y_clean, y_corrupted)

print(f"Clean Data   -> MSE: {mse_clean:.4f} | MAE: {mae_clean:.4f}")
print(f"With Outlier -> MSE: {mse_dirty:.4f} | MAE: {mae_dirty:.4f}")
print(f"MSE amplification factor: {mse_dirty / mse_clean:.1f}x")
print(f"MAE amplification factor: {mae_dirty / mae_clean:.1f}x")

## 2. Huber Loss: The Adaptive Hybrid

Huber Loss transitions smoothly between quadratic behavior for small errors and linear behavior for large errors, controlled by parameter $\delta$:
$$L_\delta(y, \hat{y}) = \begin{cases} \frac{1}{2}(y - \hat{y})^2 & \text{for } |y - \hat{y}| \le \delta \\ \delta |y - \hat{y}| - \frac{1}{2}\delta^2 & \text{otherwise} \end{cases}$$


In [ ]:
import numpy as np


def huber_loss(y_true: np.ndarray, y_pred: np.ndarray, delta: float = 1.35) -> float:
    residual = np.abs(y_true - y_pred)
    linear_mask = residual > delta
    quadratic_loss = 0.5 * (residual**2)
    linear_loss = delta * residual - 0.5 * (delta**2)
    losses = np.where(linear_mask, linear_loss, quadratic_loss)
    return float(np.mean(losses))


print(f"Huber Loss (Clean):   {huber_loss(y_clean, y_est, delta=1.0):.4f}")
print(f"Huber Loss (Outlier): {huber_loss(y_clean, y_corrupted, delta=1.0):.4f}")

## 3. Case Study: Evaluating Housing Predictions from `data/regression_house_prices.csv`

We benchmark two candidate models:
1. `pred_linear_k`: Ordinary least squares baseline model.
2. `pred_ridge_k`: Regularized Ridge regression model with cross-validated hyperparameters.


In [ ]:
import csv
from pathlib import Path

import numpy as np

csv_path = Path("../data/regression_house_prices.csv")
actuals = []
preds_linear = []
preds_ridge = []

with csv_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        actuals.append(float(row["actual_price_k"]))
        preds_linear.append(float(row["pred_linear_k"]))
        preds_ridge.append(float(row["pred_ridge_k"]))

y_act = np.array(actuals)
p_lin = np.array(preds_linear)
p_rdg = np.array(preds_ridge)

print(f"{'Metric':<10} | {'Linear Baseline':<16} | {'Ridge Regularized':<16}")
print("-" * 48)
print(
    f"{'MSE':<10} | {mean_squared_error(y_act, p_lin):<16.4f} | {mean_squared_error(y_act, p_rdg):<16.4f}"
)
print(
    f"{'RMSE':<10} | {root_mean_squared_error(y_act, p_lin):<16.4f} | {root_mean_squared_error(y_act, p_rdg):<16.4f}"
)
print(
    f"{'MAE':<10} | {mean_absolute_error(y_act, p_lin):<16.4f} | {mean_absolute_error(y_act, p_rdg):<16.4f}"
)
print(
    f"{'MAPE':<10} | {mean_absolute_percentage_error(y_act, p_lin):<15.2f}% | {mean_absolute_percentage_error(y_act, p_rdg):<15.2f}%"
)
print(f"{'R^2':<10} | {r2_score(y_act, p_lin):<16.4f} | {r2_score(y_act, p_rdg):<16.4f}")